# Python Logging — Complete Practical Guide

A developer-oriented notebook covering logging from fundamentals to reusable application architecture.

## Outcomes
- Understand levels, loggers, handlers and formatters.
- Configure console, file and multiple handlers.
- Use rotating files, named loggers, filters and `dictConfig()`.
- Capture tracebacks and attach request/job context.
- Apply logging to APIs, data pipelines and ML workflows.

Run examples in order. Some cells create log files in the current working directory. Practice problems intentionally contain no solutions.

## 1. Logging vs print()

Logging records application events: startup, milestones, warnings, failures, timings and diagnostic details.

| `print()` | `logging` |
|---|---|
| Mainly displays text | Records application events |
| No standard severity | DEBUG, INFO, WARNING, ERROR, CRITICAL |
| Manual routing | Console, file and custom handlers |
| No built-in rotation | Rotating file handlers |
| Hard to control at scale | Named loggers and central configuration |

Use `print()` for quick exploration. Use logging in applications, reusable modules and deployed scripts.

## 2. Logging levels

Levels represent severity. The default root logger threshold is WARNING.

In [ ]:
import logging

print(logging.DEBUG)       # 10
print(logging.INFO)        # 20
print(logging.WARNING)     # 30
print(logging.ERROR)       # 40
print(logging.CRITICAL)    # 50

| Level | Intended use | Example |
|---|---|---|
| DEBUG | Detailed diagnostics | Parsed 120 records |
| INFO | Normal milestones | Application started |
| WARNING | Unexpected, but recoverable | Retry attempt 2 |
| ERROR | Operation failed | Database query failed |
| CRITICAL | Severe failure | Required configuration unavailable |

A record must pass the logger's effective level and the handler's level.

## 3. Implementation 1 — `basicConfig()`

Quick setup for a small script. In a fresh process, it configures the root logger.

In [ ]:
import logging

logging.basicConfig(
    level=logging.DEBUG,
    format="%(asctime)s | %(levelname)s | %(message)s",
    datefmt="%Y-%m-%d %H:%M:%S"
)

logging.debug("Debug details")
logging.info("Application started")
logging.warning("Configuration missing; using default")
logging.error("Could not process operation")
logging.critical("Application cannot continue")

**Notebook caveat:** `basicConfig()` normally does nothing if the root logger already has handlers. In a standalone script, modern Python supports `force=True` to replace root handlers:

```python
logging.basicConfig(level=logging.INFO, force=True)
```

Do not configure logging repeatedly inside imported library modules. Libraries should create named loggers and let the application configure output.

## 4. Logger, Handler and Formatter

- **Logger:** Application code sends records to it.
- **Handler:** Routes accepted records to console, file, network, etc.
- **Formatter:** Controls how each record is rendered.

Flow: `Application → Logger → Handler → Formatter → Output`

In [ ]:
import logging

logger = logging.getLogger("inventory")
logger.setLevel(logging.DEBUG)
logger.propagate = False

# Remove old handlers so rerunning the cell does not duplicate output.
for old_handler in logger.handlers[:]:
    logger.removeHandler(old_handler)
    old_handler.close()

console_handler = logging.StreamHandler()
console_handler.setLevel(logging.INFO)
formatter = logging.Formatter(
    "%(asctime)s | %(name)s | %(levelname)s | %(message)s",
    datefmt="%H:%M:%S"
)
console_handler.setFormatter(formatter)
logger.addHandler(console_handler)

logger.debug("Filtered by handler threshold")
logger.info("Inventory service initialized")
logger.warning("Stock is running low")

Useful formatter fields:

| Field | Meaning |
|---|---|
| `%(asctime)s` | Human-readable time |
| `%(name)s` | Logger name |
| `%(levelname)s` | Severity |
| `%(message)s` | Final message |
| `%(filename)s` | Source filename |
| `%(lineno)d` | Source line |
| `%(funcName)s` | Function name |
| `%(process)d` | Process ID |
| `%(threadName)s` | Thread name |

## 5. Implementation 2 — Console + file with different thresholds

INFO and above go to console; DEBUG and above go to file.

In [ ]:
import logging

logger = logging.getLogger("order_service")
logger.setLevel(logging.DEBUG)
logger.propagate = False

for old_handler in logger.handlers[:]:
    logger.removeHandler(old_handler)
    old_handler.close()

console = logging.StreamHandler()
console.setLevel(logging.INFO)

file_handler = logging.FileHandler("orders.log", encoding="utf-8")
file_handler.setLevel(logging.DEBUG)

formatter = logging.Formatter(
    "%(asctime)s | %(levelname)-8s | %(name)s | %(message)s"
)
console.setFormatter(formatter)
file_handler.setFormatter(formatter)
logger.addHandler(console)
logger.addHandler(file_handler)

logger.debug("Order validation started")
logger.info("Order ORD-1001 accepted")
logger.warning("Delivery address needs verification")
logger.error("Payment authorization failed for ORD-1003")

In [ ]:
from pathlib import Path

print(Path("orders.log").resolve())
print(Path("orders.log").read_text(encoding="utf-8"))

## 6. Implementation 3 — File rotation

A plain `FileHandler` grows indefinitely. Rotation limits file size or age.

### 6.1 Size-based rotation — `RotatingFileHandler`

In [ ]:
import logging
from logging.handlers import RotatingFileHandler

logger = logging.getLogger("size_rotation")
logger.setLevel(logging.INFO)
logger.propagate = False

for h in logger.handlers[:]:
    logger.removeHandler(h)
    h.close()

handler = RotatingFileHandler(
    "application_size.log",
    maxBytes=2_000,       # Small limit for demonstration
    backupCount=3,
    encoding="utf-8"
)
handler.setFormatter(logging.Formatter(
    "%(asctime)s | %(levelname)s | %(message)s"
))
logger.addHandler(handler)

for number in range(100):
    logger.info("Processing batch %03d with diagnostic details", number)

### 6.2 Time-based rotation — `TimedRotatingFileHandler`

In [ ]:
import logging
from logging.handlers import TimedRotatingFileHandler

logger = logging.getLogger("time_rotation")
logger.setLevel(logging.INFO)
logger.propagate = False

for h in logger.handlers[:]:
    logger.removeHandler(h)
    h.close()

handler = TimedRotatingFileHandler(
    "daily_application.log",
    when="midnight",
    interval=1,
    backupCount=7,
    encoding="utf-8",
    utc=False
)
handler.setFormatter(logging.Formatter(
    "%(asctime)s | %(levelname)s | %(message)s"
))
logger.addHandler(handler)
logger.info("Daily log rotation is configured")

Common time values include `S`, `M`, `H`, `D`, `midnight` and weekday names such as `W0`. Use size rotation when volume is the main concern and time rotation for daily/weekly operational reporting.

**Production note:** Multiple processes writing to the same rotating file can cause coordination problems. Prefer centralized logging, platform-collected stdout/stderr or a logging service for multi-process systems.

## 7. Multiple named loggers in a multi-module project

Use `logging.getLogger(__name__)` in each module. Logger names follow the package structure.

Suggested structure:

```text
shop_app/
    main.py
    logging_config.py
    services/
        payment.py
        inventory.py
```

### `logging_config.py`

In [ ]:
import logging
from logging.handlers import RotatingFileHandler

def configure_logging():
    root = logging.getLogger()
    root.setLevel(logging.DEBUG)

    if root.handlers:
        return

    formatter = logging.Formatter(
        "%(asctime)s | %(levelname)-8s | %(name)s | %(filename)s:%(lineno)d | %(message)s"
    )

    console = logging.StreamHandler()
    console.setLevel(logging.INFO)
    console.setFormatter(formatter)

    file_handler = RotatingFileHandler(
        "shop_application.log",
        maxBytes=5_000_000,
        backupCount=5,
        encoding="utf-8"
    )
    file_handler.setLevel(logging.DEBUG)
    file_handler.setFormatter(formatter)

    root.addHandler(console)
    root.addHandler(file_handler)

### `services/payment.py`

In [ ]:
import logging

logger = logging.getLogger(__name__)

def process_payment(order_id, amount):
    logger.info("Starting payment order_id=%s", order_id)
    if amount <= 0:
        logger.warning("Rejected non-positive amount order_id=%s", order_id)
        raise ValueError("Payment amount must be positive")

    # Replace this simulated operation with a payment provider call.
    logger.debug("Provider request prepared order_id=%s", order_id)
    logger.info("Payment completed order_id=%s", order_id)
    return {"order_id": order_id, "status": "paid"}

### `services/inventory.py`

In [ ]:
import logging

logger = logging.getLogger(__name__)

def reserve_stock(product_id, quantity, available_stock):
    logger.info("Reservation requested product_id=%s", product_id)
    if quantity <= 0:
        logger.warning("Invalid quantity=%s", quantity)
        raise ValueError("Quantity must be positive")
    if quantity > available_stock:
        logger.error(
            "Insufficient stock product_id=%s requested=%s available=%s",
            product_id, quantity, available_stock
        )
        return False
    logger.info("Stock reserved product_id=%s quantity=%s", product_id, quantity)
    return True

### `main.py`

In [ ]:
import logging
from logging_config import configure_logging
from services.payment import process_payment
from services.inventory import reserve_stock

configure_logging()
logger = logging.getLogger(__name__)

def main():
    logger.info("Shop application started")
    reserve_stock("SKU-501", 2, 10)
    process_payment("ORD-2001", 1499.00)
    logger.info("Shop application finished")

if __name__ == "__main__":
    main()

Put each snippet in its corresponding file and run `python main.py` from the project root.

**Architecture rule:** Configure handlers once at application startup. Modules create named loggers and emit records. Configuring handlers in every module can cause duplicate output through propagation.

## 8. Exception logging and tracebacks

Use `logger.exception()` inside an `except` block when you want the exception traceback included.

In [ ]:
import logging

logger = logging.getLogger("exception_demo")
logger.setLevel(logging.DEBUG)
logger.propagate = False

if not logger.handlers:
    handler = logging.StreamHandler()
    handler.setFormatter(logging.Formatter("%(levelname)s | %(message)s"))
    logger.addHandler(handler)

def load_configuration(path):
    try:
        with open(path, "r", encoding="utf-8") as file:
            return file.read()
    except FileNotFoundError:
        logger.exception("Configuration file could not be loaded: %s", path)
        raise

try:
    load_configuration("missing_config.json")
except FileNotFoundError:
    print("The caller handled the exception.")

Other options:
- `logger.error("Operation failed", exc_info=True)` includes the active exception traceback.
- `logger.exception("Operation failed")` is a convenient ERROR-level call inside an exception handler.
- `stack_info=True` includes the current stack; it is not the same as an exception traceback.

Log at the layer with enough context. Then handle the error or re-raise it; do not silently swallow every exception.

## 9. Parameterized messages and useful log content

Prefer parameterized logging over f-strings for routine messages.

In [ ]:
import logging

logger = logging.getLogger("formatting_demo")
logger.setLevel(logging.DEBUG)
if not logger.handlers:
    logger.addHandler(logging.StreamHandler())

order_id = "ORD-3001"
total = 2599.50

# Preferred: interpolation occurs only if the record is emitted.
logger.info("Order %s total is %.2f", order_id, total)

# Valid, but string formatting happens before the logger checks the level.
logger.info(f"Order {order_id} total is {total:.2f}")

Useful logs answer: what operation happened, which safe identifier was involved, what was the outcome, and what measurement or reason matters.

Never log passwords, access tokens, full payment-card data or unnecessary personal information. Prefer safe identifiers and redact sensitive values.

## 10. Contextual logging with `extra` and `LoggerAdapter`

Custom context connects related events. The formatter must know which custom fields to display.

In [ ]:
import logging

logger = logging.getLogger("request_demo")
logger.setLevel(logging.INFO)
logger.propagate = False

for h in logger.handlers[:]:
    logger.removeHandler(h)
    h.close()

handler = logging.StreamHandler()
handler.setFormatter(logging.Formatter(
    "%(levelname)s | request_id=%(request_id)s | %(message)s"
))
logger.addHandler(handler)

logger.info("Request accepted", extra={"request_id": "req-7f91"})

### `LoggerAdapter` for repeated context

In [ ]:
import logging

logger = logging.getLogger("worker_demo")
logger.setLevel(logging.INFO)
logger.propagate = False

if not logger.handlers:
    handler = logging.StreamHandler()
    handler.setFormatter(logging.Formatter(
        "%(levelname)s | job_id=%(job_id)s | %(message)s"
    ))
    logger.addHandler(handler)

job_log = logging.LoggerAdapter(logger, {"job_id": "JOB-804"})
job_log.info("Job started")
job_log.warning("One record was skipped")
job_log.info("Job completed")

Custom `extra` fields must not collide with built-in `LogRecord` attributes. If a formatter expects a field that a record does not supply, formatting can fail. Use a consistent schema or a filter/adapter that supplies defaults.

## 11. Centralized configuration with `dictConfig()`

For larger applications, `logging.config.dictConfig()` keeps setup declarative.

In [ ]:
import logging
import logging.config

LOGGING_CONFIG = {
    "version": 1,
    "disable_existing_loggers": False,
    "formatters": {
        "standard": {
            "format": "%(asctime)s | %(levelname)s | %(name)s | %(message)s"
        }
    },
    "handlers": {
        "console": {
            "class": "logging.StreamHandler",
            "level": "INFO",
            "formatter": "standard",
            "stream": "ext://sys.stdout"
        },
        "file": {
            "class": "logging.handlers.RotatingFileHandler",
            "level": "DEBUG",
            "formatter": "standard",
            "filename": "dict_config_app.log",
            "maxBytes": 2_000_000,
            "backupCount": 4,
            "encoding": "utf-8"
        }
    },
    "loggers": {
        "data_pipeline": {
            "level": "DEBUG",
            "handlers": ["console", "file"],
            "propagate": False
        }
    },
    "root": {
        "level": "WARNING",
        "handlers": ["console"]
    }
}

logging.config.dictConfig(LOGGING_CONFIG)
pipeline_logger = logging.getLogger("data_pipeline")
pipeline_logger.debug("Reading source data")
pipeline_logger.info("Data validation completed")
pipeline_logger.error("One input file failed validation")

| Configuration | Best fit |
|---|---|
| `basicConfig()` | Small script or prototype |
| Explicit logger/handler objects | Custom or dynamic setup |
| `dictConfig()` | Larger applications |
| `fileConfig()` | Existing INI-based configuration |

Reusable packages should create module-level named loggers but should not configure the root logger.

## 12. Filters — selecting records

A filter inspects a record and decides whether it should be emitted.

In [ ]:
import logging

class OnlyPaymentEvents(logging.Filter):
    def filter(self, record):
        return "payment" in record.getMessage().lower()

logger = logging.getLogger("filter_demo")
logger.setLevel(logging.DEBUG)
logger.propagate = False

for h in logger.handlers[:]:
    logger.removeHandler(h)
    h.close()

handler = logging.StreamHandler()
handler.addFilter(OnlyPaymentEvents())
handler.setFormatter(logging.Formatter("%(levelname)s | %(message)s"))
logger.addHandler(handler)

logger.info("Payment authorized")
logger.info("Inventory updated")
logger.error("Payment gateway unavailable")

## 13. Real-world example — API request lifecycle

This simulates an API endpoint. It records request ID, route, status, duration and failure context; it does not make a network request.

In [ ]:
import logging
import time
import uuid

logger = logging.getLogger("api_service")
logger.setLevel(logging.INFO)
logger.propagate = False

if not logger.handlers:
    handler = logging.StreamHandler()
    handler.setFormatter(logging.Formatter(
        "%(asctime)s | %(levelname)s | %(message)s"
    ))
    logger.addHandler(handler)

def get_customer(customer_id):
    request_id = str(uuid.uuid4())[:8]
    started = time.perf_counter()
    logger.info(
        "request_started request_id=%s method=GET path=/customers/%s",
        request_id, customer_id
    )

    try:
        if customer_id <= 0:
            raise ValueError("customer_id must be positive")

        customer = {"customer_id": customer_id, "name": "Sample Customer"}
        duration_ms = (time.perf_counter() - started) * 1000
        logger.info(
            "request_completed request_id=%s status=200 duration_ms=%.2f",
            request_id, duration_ms
        )
        return customer

    except ValueError:
        duration_ms = (time.perf_counter() - started) * 1000
        logger.exception(
            "request_failed request_id=%s status=400 duration_ms=%.2f",
            request_id, duration_ms
        )
        raise

print(get_customer(101))
try:
    get_customer(-1)
except ValueError:
    print("Caller received the validation error.")

In a real web framework, middleware commonly creates the request ID and passes it into service functions using an adapter or context variables. Do not log complete request bodies by default; they may contain credentials or personal data.

## 14. Real-world example — data pipeline and ML training logs

Record meaningful stages, counts, durations and metrics rather than dumping datasets into logs.

In [ ]:
import logging
import time
from pathlib import Path

logger = logging.getLogger("ml_pipeline")
logger.setLevel(logging.INFO)
logger.propagate = False

if not logger.handlers:
    handler = logging.StreamHandler()
    handler.setFormatter(logging.Formatter(
        "%(asctime)s | %(levelname)s | %(name)s | %(message)s"
    ))
    logger.addHandler(handler)

def run_training_pipeline(dataset_path, model_name):
    started = time.perf_counter()
    logger.info("pipeline_started model=%s dataset=%s", model_name, dataset_path)

    try:
        path = Path(dataset_path)
        if not path.exists():
            raise FileNotFoundError(f"Dataset not found: {path}")

        logger.info("dataset_loaded file_size_bytes=%s", path.stat().st_size)
        logger.info("preprocessing_started")
        time.sleep(0.1)
        logger.info("preprocessing_completed rows=%d features=%d", 1200, 18)

        logger.info("training_started algorithm=%s", model_name)
        time.sleep(0.1)
        logger.info("training_completed")

        metrics = {"accuracy": 0.93, "f1_score": 0.91}
        logger.info(
            "evaluation_completed accuracy=%.4f f1_score=%.4f",
            metrics["accuracy"], metrics["f1_score"]
        )
        logger.info(
            "pipeline_completed duration_seconds=%.2f",
            time.perf_counter() - started
        )
        return metrics

    except Exception:
        logger.exception("pipeline_failed model=%s", model_name)
        raise

Path("sample_dataset.csv").write_text(
    "feature,target\n1,0\n2,1\n", encoding="utf-8"
)
run_training_pipeline("sample_dataset.csv", "RandomForestClassifier")

**ML logging checklist**
- Dataset/run ID, schema version, row and feature counts.
- Random seed, model type, hyperparameters and config/code version.
- Training duration, evaluation metrics and artifact location.
- Warnings for data-quality thresholds.
- No raw records, credentials or sensitive feature values.

Experiment tracking tools can complement logs; they do not replace runtime diagnostics.

## 15. Structured logging — JSON output

JSON records are useful for log collectors. This example uses the standard library.

In [ ]:
import json
import logging
from datetime import datetime, timezone

class JsonFormatter(logging.Formatter):
    def format(self, record):
        payload = {
            "timestamp": datetime.fromtimestamp(
                record.created, tz=timezone.utc
            ).isoformat(),
            "level": record.levelname,
            "logger": record.name,
            "message": record.getMessage(),
            "module": record.module,
            "line": record.lineno,
        }

        for key in ("request_id", "job_id", "customer_id"):
            if hasattr(record, key):
                payload[key] = getattr(record, key)

        if record.exc_info:
            payload["exception"] = self.formatException(record.exc_info)

        return json.dumps(payload, ensure_ascii=False)

logger = logging.getLogger("json_demo")
logger.setLevel(logging.INFO)
logger.propagate = False

for h in logger.handlers[:]:
    logger.removeHandler(h)
    h.close()

handler = logging.StreamHandler()
handler.setFormatter(JsonFormatter())
logger.addHandler(handler)

logger.info(
    "Order processed",
    extra={"request_id": "req-102", "customer_id": "cust-88"}
)

For nested context, async correlation or platform integrations, consider a maintained structured-logging library. Keep the JSON schema stable so dashboards and alerts remain compatible.

## 16. Logger hierarchy, propagation and duplicate logs

Names form a hierarchy: `app`, `app.api`, `app.api.orders`. Child records normally propagate to parent loggers and eventually the root logger.

In [ ]:
import logging

root = logging.getLogger()
root.setLevel(logging.WARNING)

parent = logging.getLogger("demo_app")
child = logging.getLogger("demo_app.orders")

print("Child name:", child.name)
print("Child propagates:", child.propagate)
print("Parent name:", child.parent.name)
print("Effective level:", child.getEffectiveLevel())

**Avoid duplicate output:**
1. Configure handlers centrally, usually at application startup.
2. Let module loggers propagate to root unless there is a reason not to.
3. If a logger owns handlers, consider `logger.propagate = False`.
4. Do not attach the same handler repeatedly.
5. Do not configure logging independently in every imported module.

Useful checks:

```python
logger = logging.getLogger("app.api")
print(logger.handlers)
print(logger.propagate)
print(logger.getEffectiveLevel())
```

## 17. Production logging checklist

- Use `logging.getLogger(__name__)` in modules.
- Configure logging once at application startup.
- In containers, prefer stdout/stderr collected by the platform.
- Use rotation for application-owned local log files.
- Choose sensible levels for development and production.
- Include timestamps, severity, logger name and useful identifiers.
- Use `logger.exception()` when a traceback is useful.
- Add request, job or experiment IDs to connect events.
- Never log secrets, tokens, passwords or unnecessary personal data.
- Avoid huge objects, entire dataframes and high-volume repetitive events.
- Prefer parameterized messages: `logger.info("Loaded %d rows", count)`.
- Define retention, access control and deletion policies.
- Use metrics and tracing alongside logs; logs alone are not complete observability.

## 18. Common mistakes and fixes

| Mistake | Better approach |
|---|---|
| `print()` throughout deployed code | Use severity-based logging |
| `basicConfig()` in every module | Configure centrally once |
| Add handlers every function call | Add handlers once at startup |
| Log exception and silently hide it | Handle deliberately or re-raise |
| ERROR for every unusual event | Select severity based on impact |
| Log passwords or tokens | Redact or omit sensitive values |
| Dump entire dataframes | Log counts and safe summaries |
| One unlimited log file | Rotate or use platform collection |
| F-strings for every log call | Prefer parameterized messages |
| Handler in every module | Use hierarchy and central handlers |

## 19. Advanced practice — implement without solutions

Each task combines logging concepts with realistic application requirements. Code cells are intentionally blank.

### Problem 1 — Production Logging System for an E-commerce Application

Build reusable logging configuration for an e-commerce backend.

**Requirements**
- Create `logging_config.py` with `configure_logging()`.
- Use `dictConfig()` or explicit logger/handler/formatter objects.
- Send INFO and above to console; DEBUG and above to a rotating file.
- Make maximum file size and backup count configurable.
- Include timestamp, level, logger name, module, line number and message.
- Create named loggers for orders, payments, inventory and authentication.
- Prevent duplicate records if setup is called twice.
- Demonstrate DEBUG appearing in the file but not on console.

**Acceptance criteria**
- Modules use `logging.getLogger(__name__)`.
- Service modules do not configure root.
- One application-level setup initializes logging.
- Rotation retains the configured number of backups.

### Problem 2 — API Request Logging and Correlation IDs

Build a request-processing simulation.

**Requirements**
- Generate a request ID for every request.
- Log request start, route, method, completion status and duration.
- Use `LoggerAdapter` or `extra` to attach request ID consistently.
- Implement success, validation-error and unexpected-error paths.
- Include tracebacks for unexpected exceptions.
- Distinguish concurrent or repeated requests.
- Do not log passwords, tokens or complete request payloads.

**Acceptance criteria**
- Each request lifecycle is traceable by request ID.
- Duration uses `time.perf_counter()`.
- Caller receives the correct result or exception.
- Error logs contain useful investigation context.

### Problem 3 — Data Pipeline and ML Experiment Logger

Build a logger for a machine-learning workflow.

**Requirements**
- Track experiment ID, dataset ID, model name and random seed.
- Log loading, validation, preprocessing, training, evaluation and artifact saving.
- Record row count, feature count, missing-value count, duration and metrics.
- Warn when data-quality thresholds are exceeded.
- Log exceptions with tracebacks when a stage fails.
- Save logs to a rotating file and show milestones on console.
- Keep raw records, credentials and sensitive feature values out of logs.

**Acceptance criteria**
- Every run is identifiable by experiment ID.
- A failed stage produces a useful error record.
- Final output has a clear completion or failure event.
- Logger can be reused with different model classes.

### Problem 4 — Background Job Monitor with Filters and Multiple Handlers

Create a background job processor.

**Requirements**
- Use a named logger for the job system.
- Console handler: WARNING and above.
- File handler: DEBUG and above.
- Add a filter that sends only a selected job category to a dedicated handler.
- Include job ID, attempt number, status and elapsed time.
- Simulate retry behavior for temporary failures.
- Log final outcome after success or exhausted retries.
- Avoid duplicates caused by propagation.

**Acceptance criteria**
- Console and file thresholds differ.
- Category filtering works independently of severity.
- Retry attempts can be distinguished.
- A job's complete lifecycle is traceable.

### Problem 5 — Reusable Logging Package for a Multi-Module Project

Suggested structure:

```text
my_logging/
    __init__.py
    config.py
    context.py
    formatters.py
    filters.py
application/
    main.py
    services/
        billing.py
        reporting.py
        notifications.py
```

**Requirements**
- Implement one central configuration entry point.
- Support console, rotating-file and JSON output modes.
- Let callers configure levels and log directory.
- Provide reusable request-ID/job-ID context.
- Handle optional custom fields safely in the formatter.
- Filter noisy health-check events.
- Importing the package must not configure logging automatically.
- Document initialization and usage.

**Acceptance criteria**
- Configuration is explicit and repeatable.
- Application modules do not create handlers.
- JSON output is valid for every emitted record.
- Missing optional context does not break formatting.
- Importing package does not modify root logger.

## 20. Final revision checklist

- [ ] Explain why logging is better than `print()` for application diagnostics.
- [ ] Explain five standard levels and their intended use.
- [ ] Distinguish Logger, Handler and Formatter.
- [ ] Explain `basicConfig()` and notebook behavior.
- [ ] Configure console and file handlers with different thresholds.
- [ ] Distinguish `FileHandler`, `RotatingFileHandler` and `TimedRotatingFileHandler`.
- [ ] Explain named loggers, hierarchy and propagation.
- [ ] Diagnose and prevent duplicate logs.
- [ ] Capture tracebacks with `logger.exception()`.
- [ ] Attach context with `extra` and `LoggerAdapter`.
- [ ] Configure filters and `dictConfig()`.
- [ ] Design safe logs for APIs, data pipelines and ML systems.

**Recommended order:** Complete Problems 1 and 2 first. Apply Problem 3 to one of your ML projects. Finish with Problems 4 and 5 for reusable architecture.